## Credit Risk Data Cleaning/Preprocessing

**Project:** End-to-End Bank Credit Risk for Financial Services  
**Phase:** 1 of 6 - Data Cleaning & Preprocessing  
**Author:** Sunmi  
**Dataset:** Credit Risk Dataset (32,581 initial records)

---

## Executive Summary

This notebook documents the data cleaning process for a bank credit risk prediction project. The dataset contains information about 32,581 bank customers.The target variable indicates whether a loan applicant defaults.

**Key Cleaning Operations:**
- Removed 165 duplicate customer records
- Handled missing values across features using statistical imputation
- Standardized data types for modeling compatibility
- Renamed the column to be better understood during EDA and feature engineeering.

**Final Output:** A clean dataset of 32,416 unique customers ready for exploratory data analysis and feature engineering.

---
## 1. Environment Setup

We begin by importing the necessary Python libraries for data manipulation and analysis. Warnings are suppressed to maintain clean output throughout the notebook.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

---
## 2. Data Loading & Initial Inspection

The raw dataset is loaded from CSV format. This initial inspection helps us understand the data structure, identify potential quality issues, and plan our cleaning strategy.

In [2]:
df=pd.read_csv(r"c:\Users\HP USER\Desktop\Credit Risk Full Project\credit_risk_dataset.csv.zip")

In [3]:
df.shape

(32581, 12)

The dataset contains **32,581 rows** (customer records) and **12 columns** (features). This is our starting point before any cleaning operations.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [ ]:
df.head(10)

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
5,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2
6,26,77100,RENT,8.0,EDUCATION,B,35000,12.42,1,0.45,N,3
7,24,78956,RENT,5.0,MEDICAL,B,35000,11.11,1,0.44,N,4
8,24,83000,RENT,8.0,PERSONAL,A,35000,8.90,1,0.42,N,2
9,21,10000,OWN,6.0,VENTURE,D,1600,14.74,1,0.16,N,3


Some column names might be hard to understand due to the words that are shortened so we will have to rename them to ensure clarity during analysis. 

In [6]:
df.rename(columns={
    "person_age": "Age",
    "person_income": "Income",
    "person_home_ownership": "Home_Ownership",
    "person_emp_length": "Employment_Years",
    "loan_intent": "Loan_Purpose",
    "loan_grade": "Loan_Risk_Grade",
    "loan_amnt": "Loan_Amount",
    "loan_int_rate": "Interest_Rate",
    "loan_status": "Loan_Default_Status",
    "loan_percent_income": "Loan_Income_Ratio",
    "cb_person_default_on_file": "Previous_Default",
    "cb_person_cred_hist_length": "Credit_History_Years"
}, inplace=True)

#### Renaming Columns

The column names were renamed to make them more clear and easier to understand, as the original names contained abbreviations that could be confusing during analysis.

Checking if the columns were successfuly renamed

In [7]:
df.head()

,Age,Income,Home_Ownership,Employment_Years,Loan_Purpose,Loan_Risk_Grade,Loan_Amount,Interest_Rate,Loan_Default_Status,Loan_Income_Ratio,Previous_Default,Credit_History_Years
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


**Initial Observations:**
- No visible Missing values across the columns in the dataset
- The dataset contains features related to customer personal and loan information
- Some columns names are not quite understandable
- Some columns contains visible outliers(e.g Employment_Years)

---
## 3. Handling Duplicate Records

Duplicate customer records can skew our analysis and model training. We identify and remove exact duplicates across all columns to ensure each customer appears only once in our dataset.

**Business Context:** In banking datasets, duplicates may arise from data entry errors, system migrations, or multiple account snapshots. Removing them ensures our credit risk analysis reflects accurate inputed data.

checking the total duplicates

In [8]:
df.duplicated().sum()

np.int64(165)

The rows that are duplicated in the dataset is a total of `165` rows.
Let's view them.

In [9]:
df[df.duplicated()]

,Age,Income,Home_Ownership,Employment_Years,Loan_Purpose,Loan_Risk_Grade,Loan_Amount,Interest_Rate,Loan_Default_Status,Loan_Income_Ratio,Previous_Default,Credit_History_Years
15975,23,42000,RENT,5.0,VENTURE,B,6000,9.99,0,0.14,N,4
15989,23,90000,MORTGAGE,7.0,EDUCATION,B,8000,10.36,0,0.09,N,3
15995,24,48000,MORTGAGE,4.0,MEDICAL,A,4000,5.42,0,0.08,N,4
16025,24,10000,RENT,8.0,PERSONAL,A,3000,7.90,1,0.30,N,3
16028,23,100000,MORTGAGE,7.0,EDUCATION,A,15000,7.88,0,0.15,N,4
...,...,...,...,...,...,...,...,...,...,...,...,...
32010,42,39996,MORTGAGE,2.0,HOMEIMPROVEMENT,A,2500,5.42,0,0.06,N,12
32047,36,250000,RENT,2.0,DEBTCONSOLIDATION,A,20000,7.88,0,0.08,N,17
32172,49,120000,MORTGAGE,12.0,MEDICAL,B,12000,10.99,0,0.10,N,12
32259,39,40000,OWN,4.0,VENTURE,B,1000,10.37,0,0.03,N,16


Now we drop the duplicates

In [10]:
df.drop_duplicates(inplace=True)

We need to be aware of how many datasets we have left 

In [11]:
df.shape

(32416, 12)

**Result:** The dataset now contains **32,416 unique records** (reduced from 32,581), indicating that **165 duplicate rows** were successfully removed.

---
## 4. Missing Value Analysis

Let's examine the current state of the data more closely. Understanding the distribution and characteristics of our features helps inform the best approach for handling missing values.

Checking the total number of misssing values in each column

In [12]:
df.isna().sum()

Age                        0
Income                     0
Home_Ownership             0
Employment_Years         887
Loan_Purpose               0
Loan_Risk_Grade            0
Loan_Amount                0
Interest_Rate           3095
Loan_Default_Status        0
Loan_Income_Ratio          0
Previous_Default           0
Credit_History_Years       0
dtype: int64

Checking the percentage of missing values in each column 

In [13]:
df.isnull().sum()/len(df)*100

Age                     0.000000
Income                  0.000000
Home_Ownership          0.000000
Employment_Years        2.736303
Loan_Purpose            0.000000
Loan_Risk_Grade         0.000000
Loan_Amount             0.000000
Interest_Rate           9.547754
Loan_Default_Status     0.000000
Loan_Income_Ratio       0.000000
Previous_Default        0.000000
Credit_History_Years    0.000000
dtype: float64

---
## 5. Missing Value Imputation

Given the minimal number of missing values in the rows, we'll use statistical imputation methods that preserve the distribution characteristics of each feature:

**Numeric Features:** Use **median imputation** rather than mean to reduce the impact of potential outliers.


In [14]:
df['Interest_Rate']=df['Interest_Rate'].fillna(df['Interest_Rate'].median())
df['Employment_Years']=df['Employment_Years'].fillna(df['Employment_Years'].median())


#### Checking if all null values has been handled

In [15]:
df.isna().sum()

Age                     0
Income                  0
Home_Ownership          0
Employment_Years        0
Loan_Purpose            0
Loan_Risk_Grade         0
Loan_Amount             0
Interest_Rate           0
Loan_Default_Status     0
Loan_Income_Ratio       0
Previous_Default        0
Credit_History_Years    0
dtype: int64

All null values has been successfully handled

---
## 6. Data Type Standardization
After imputation, we convert numeric features to their appropriate integer data types. This is essential for:
- **Data integrity:** Years are conceptually integers, not decimals

#### Employment_Years Conversion 

In [16]:
df['Employment_Years']=df['Employment_Years'].astype(int)

Employment_Years is now stored as integers

---
## 7. Encoding Previous Default Column


In [17]:
df["Previous_Default"] = df["Previous_Default"].map({"Y": "Yes", "N": "No"})

The `Previous_Default` column was updated from Y/N values to Yes/No for better readability and understanding during analysis.

To check if the the column was successfully updated without mistakenly creating missing values

In [18]:
df["Previous_Default"].value_counts() #to check if the the column was successfully updated without mistakenly creating missing values

Previous_Default
No     26686
Yes     5730
Name: count, dtype: int64

Fixing few features in `Loan_Purpose`

In [19]:
df["Loan_Purpose"] = df["Loan_Purpose"].replace({"HOMEIMPROVEMENT":"HOME IMPROVEMENT", "DEBTCONSOLIDATION": "DEBT CONSOLIDATION"})

---
## 8. Final Data Validation

Let's verify that our cleaning operations were successful by examining the final state of the dataset.

In [20]:
df.head(20)

,Age,Income,Home_Ownership,Employment_Years,Loan_Purpose,Loan_Risk_Grade,Loan_Amount,Interest_Rate,Loan_Default_Status,Loan_Income_Ratio,Previous_Default,Credit_History_Years
0,22,59000,RENT,123,PERSONAL,D,35000,16.02,1,0.59,Yes,3
1,21,9600,OWN,5,EDUCATION,B,1000,11.14,0,0.10,No,2
2,25,9600,MORTGAGE,1,MEDICAL,C,5500,12.87,1,0.57,No,3
3,23,65500,RENT,4,MEDICAL,C,35000,15.23,1,0.53,No,2
4,24,54400,RENT,8,MEDICAL,C,35000,14.27,1,0.55,Yes,4
5,21,9900,OWN,2,VENTURE,A,2500,7.14,1,0.25,No,2
6,26,77100,RENT,8,EDUCATION,B,35000,12.42,1,0.45,No,3
7,24,78956,RENT,5,MEDICAL,B,35000,11.11,1,0.44,No,4
8,24,83000,RENT,8,PERSONAL,A,35000,8.90,1,0.42,No,2
9,21,10000,OWN,6,VENTURE,D,1600,14.74,1,0.16,No,3


---
## 9. Export Clean Dataset

The cleaned dataset is now ready for the next phase of the project. We export it to CSV format for use in exploratory data analysis and subsequent modeling phases.

In [21]:
df.to_csv('Credit_Risk_Dataset_Cleaned.csv', index=False)

In [22]:
df.shape

(32416, 12)

**Output:** `Credit_Risk_Dataset_Cleaned.csv`  
**Records:** 32,416 unique customers  
**Features:** 12 columns (all complete, no missing values)